In [ ]:
%load_ext nb_mypy

# Swapping Knights

This notebook solves the following problem: ![problem](knight.png).  

It is played on a heavily restricted chess board that consists of
only ten squares.  We use the usual chess notation: the columns are named `a`, `b`, `c`, `d` from left to
right and the rows are numbered `1` to `4` from bottom to top.  Initially, there are two white knights on the
squares `b4` and `c2` and two black knights on the squares `a1` and `c1`:
```
4   W
3   . .
2   . W .
1 B . B .
  a b c d
```
Here `W` denotes a white knight, `B` denotes a black knight, and `.` denotes an empty square.  The task is to
swap the white and the black knights: at the end, the white knights have to stand on the squares where the
black knights stood initially and vice versa.  The following rules apply:
- The knights move like ordinary knights in chess.  As no knight may be captured, a knight can only jump to an
  empty square.
- White and black do not have to alternate: the same side may move several times in a row.

## The Board

A square is represented as a string that consists of the letter of its column followed by the digit of its
row.  The set `Squares` contains the ten squares of the board.  The layout of the set literal below mirrors
the shape of the board.

In [ ]:
Square = str

Squares: set[Square] = {       'b4',
                               'b3', 'c3',
                               'b2', 'c2', 'd2',
                         'a1', 'b1', 'c1', 'd1'
                       }

The function `is_knight_move` takes two squares `s` and `t` and returns `True` if a knight can jump from `s` to
`t`.  A knight jumps two squares in one direction and one square in the orthogonal direction.  Hence, either the
column changes by one and the row by two, or the column changes by two and the row by one.  `dx` is the distance
between the columns of `s` and `t`, `dy` is the distance between their rows.

In [ ]:
def is_knight_move(s: Square, t: Square) -> bool:
    dx = abs(ord(s[0]) - ord(t[0]))
    dy = abs(int(s[1]) - int(t[1]))
    return {dx, dy} == {1, 2}

For every square `s` of the board, `Jumps[s]` is the set of those squares of the board that a knight standing on
`s` can jump to.  On this small board, every square has at most three such squares.

In [ ]:
Jumps: dict[Square, set[Square]]
Jumps = { s: { t for t in Squares if is_knight_move(s, t) } for s in Squares }

In [ ]:
Jumps['b3']

## States

A state has to specify which squares are occupied by the white knights and which squares are occupied by the
black knights.  The two knights of the same colour can't be distinguished.  Therefore, we represent the squares
occupied by the knights of one colour as a *set* of two squares.  A state is a pair `(White, Black)` where
- `White` is the set of the squares occupied by the white knights and
- `Black` is the set of the squares occupied by the black knights.

As sets are unordered, every placement of the knights is represented by exactly one state.  If we distinguished
the two knights of each colour, e.g. by representing their squares as tuples, every placement would be
represented by $2 \cdot 2 = 4$ states and the search space would be four times as big.

States have to be elements of sets and keys of dictionaries, so they have to be hashable.  Therefore, we use the
immutable type `frozenset` instead of `set`.  `Knights` is the type of the set of squares occupied by the knights
of one colour.  This type does not express that such a set always contains exactly two squares.

In [ ]:
Knights = frozenset[Square]
State   = tuple[Knights, Knights]

The function `next_states` takes a `state` and computes the set of all states that can be reached from `state`
by a single jump of one knight.  `Occupied` is the set of squares occupied by a knight.  A knight standing
on the square `s` can jump to every square in `Jumps[s] - Occupied`.  If a white knight jumps from `s` to `t`,
the white knights occupy the squares `White - {s} | {t}` afterwards: the square `s` is removed and the square `t`
is added.  The operator `-` binds stronger than `|`, and since the left operand `White` is a frozen set, the
result is a frozen set, too.  The first set comprehension collects the states that result from a jump of a white
knight, the second one collects the states that result from a jump of a black knight.  As the same side may move
several times in a row, a state doesn't need to record whose turn it is.

In [ ]:
def next_states(state: State) -> set[State]:
    White, Black = state
    Occupied     = White | Black
    return ({ (White - {s} | {t}, Black) for s in White for t in Jumps[s] - Occupied } |
            { (White, Black - {s} | {t}) for s in Black for t in Jumps[s] - Occupied })

Initially, the white knights stand on `b4` and `c2`, while the black knights stand on `a1` and `c1`.
In the goal state, the colours are swapped.  Since the knights of the same colour can't be distinguished,
the goal state is also the goal of the lax formulation of the puzzle: wherever a black knight stood at the
beginning, a white knight has to stand at the end, and vice versa.

In [ ]:
start: State = (frozenset({'b4', 'c2'}), frozenset({'a1', 'c1'}))
goal:  State = (frozenset({'a1', 'c1'}), frozenset({'b4', 'c2'}))

In the start state, only the black knights can move and both of them can only jump to `b3`.

In [ ]:
next_states(start)

As there are 10 positions, there are $\binom{10}{2}$ ways to choose two of these positions for the white knights.  Given a placement of the white knights, there are 8 positions left for the two black knights.  Hence
there are $\binom{10}{2} \cdot \binom{8}{2} = 45 \cdot 28 = 1260$ ways to place two white and two black knights
on the board.  The function `reachable` takes a state `start` and returns the set of all states that can be
reached from `start` by any number of moves.  `Reached` is the set of states found so far and `Frontier` is the
set of states found in the last iteration of the loop.  The loop ends when an iteration finds no new states.

In [ ]:
def reachable(start: State) -> set[State]:
    Reached:  set[State] = { start }
    Frontier: set[State] = { start }
    while Frontier:
        Frontier = { ns for s in Frontier for ns in next_states(s) } - Reached
        Reached |= Frontier
    return Reached

In [ ]:
len(reachable(start))

As there are `1260` reachable states, every placement of the knights can be reached from the start state.  
We draw the board as a graph.

## The Graph of Knight Moves

The puzzle becomes much easier to understand once the board is drawn as a graph: the nodes of this graph are the
squares and two squares are connected by an edge if a knight can jump from one square to the other.  To display
this graph, we need the module `graphviz`.

In [ ]:
import graphviz as gv

The function `board_graph` takes a `state` and returns the graph of knight moves as a `graphviz` object.  The
squares occupied by white knights are drawn with a bold border and labelled with `W`, the squares occupied by
black knights are drawn black and labelled with `B`.  As the module `graphviz` has no type annotations, we do
not specify the return type.

In [ ]:
def board_graph(state: State):
    White, Black = state
    dot = gv.Graph(engine='neato')
    for s in sorted(Squares):
        if s in White:
            dot.node(s, label=s + ': W', penwidth='3')
        elif s in Black:
            dot.node(s, label=s + ': B', style='filled', fillcolor='black', fontcolor='white')
        else:
            dot.node(s)
    for s in sorted(Squares):
        for t in sorted(Jumps[s]):
            if s < t:
                dot.edge(s, t)
    return dot

In [ ]:
board_graph(start)

The graph is a tree: the squares
$$ \texttt{b4} - \texttt{c2} - \texttt{a1} - \texttt{b3} - \texttt{d2} - \texttt{b1} - \texttt{c3} -
   \texttt{d1} - \texttt{b2}
$$
form a path and the square `c1` is a dead end that is attached to `b3`.  A knight jumps along an edge of this graph
to an empty neighbour.  Therefore, knights can never pass each other on the path.  Read from `b4`, the knights on
the path are initially ordered as white, white, black, and the second black knight waits in the dead end `c1`.
At the end, the order has to be black, black, white, and the second white knight has to be in `c1`.  The only
place where the knights can change their order is the junction `b3` together with the dead end `c1`.  This is why
the puzzle needs many moves.

## Breadth First Search

We solve the puzzle with *breadth first search*.  The two functions `path_to` and `search` below are copied from
the notebook `../Chapter-01/04-Breadth-First-Search.ipynb`.  The only change is that the type variable is called
`T` instead of `State`, since `State` is already the type of the states of this puzzle.

In [ ]:
from typing import TypeVar, Callable
T        = TypeVar('T')
NxtStFct = Callable[[T], set[T]]

Given a `state` and a parent dictionary `Parent`, the function `path_to` returns a path leading from `start` to the given `state`. Note that `parent[start] = start` and `start` is the only state with this property.

In [ ]:
def path_to(state: T, Parent: dict[T, T]) -> list[T]:
    p = Parent[state]
    if p == state:
        return [state]
    return path_to(p, Parent) + [state]

The function `search` takes three arguments to solve a *search problem*:
- `start` is the *start state* of the search problem,
- `goal` is the *goal state*, and
- `next_states` is a function with signature $\texttt{next\_states}:Q \rightarrow 2^Q$, where $Q$ is the set of states.
   For every state $s \in Q$, $\texttt{next\_states}(s)$ is the set of states that can be reached from $s$ in one step.

If successful, `search` returns a path from `start` to `goal` that is a solution of the search problem
$$ \langle Q, \texttt{next\_states}, \texttt{start}, \texttt{goal} \rangle. $$
The implementation of `search` uses the algorithm *breadth first search* to find a path from `start` to `goal`.

At the start of the $n^\textrm{th}$ iteration of the `while` loop, the following invariants are satisfied:
* `Frontier` contains exactly those states that have distance of $n-1$ from `start`.
* `Visited`  contains those states that have distance from start that is less than `n-1`.
* `Parent` is a dictionary. The keys of this dictionaries are all states from the sets `Visited`, `Frontier`, and `NewFrontier`.    Furthermore, the following invariant holds for all states $x \not= \texttt{start}$:
  - If $x = \texttt{Parent}[y]$, then $y \in \texttt{next\_states}(x)$.

At the end of every iteration, `search` prints the size of the new frontier.

In [ ]:
def search(start: T, goal: T, next_states: NxtStFct) -> list[T] | None:
    Frontier        = { start }
    Visited: set[T] = set()
    Parent          = { start: start }
    while Frontier:
        NewFrontier = set()
        for s in Frontier:
            for ns in next_states(s):
                if ns not in Visited:
                    NewFrontier.add(ns)
                    Parent[ns] = s
                    if ns == goal:
                        print("number of states: ", len(Visited) + len(Frontier) + len(NewFrontier))
                        return path_to(goal, Parent)
        Visited |= Frontier
        Frontier = NewFrontier
        print(len(Frontier))
    return None

## Solving the Puzzle

The search takes only a few milliseconds.  We don't use `%%time` here, because the extension `nb_mypy` does not
see the variables that are assigned in a cell starting with `%%time`.

In [ ]:
Path = search(start, goal, next_states)

Breadth first search explores the states in the order of their distance from `start`.  Hence, the path found is a
shortest solution: no solution needs fewer jumps.  Sets of strings are iterated in an order that depends on the
hash seed of the Python process.  Therefore, the solution found may differ from one run to the next, but its
length is always the same.

In [ ]:
assert Path is not None
print(f'The solution needs {len(Path) - 1} moves.')

## Printing the Solution

The function `board_lines` takes a `state` and returns the lines of a drawing of the board in this state, as in
the description of the puzzle at the beginning of this notebook: the rows `4` to `1` followed by a line with the
names of the columns.  The local function `symbol` returns the character that represents a given `square`:
`W` for a white knight, `B` for a black knight, `.` for an empty square, and a blank for a square that does not
belong to the board.

In [ ]:
def board_lines(state: State) -> list[str]:
    White, Black = state
    def symbol(square: Square) -> str:
        if square in White:
            return 'W'
        if square in Black:
            return 'B'
        if square in Squares:
            return '.'
        return ' '
    Rows = [row + ' ' + ' '.join(symbol(column + row) for column in 'abcd') for row in '4321']
    return [line.rstrip() for line in Rows] + ['  a b c d']

In [ ]:
print('\n'.join(board_lines(start)))

A move is a triple `(colour, source, target)`: the knight of the given `colour` jumps from the square `source` to
the square `target`.  The colour is `'W'` for white and `'B'` for black.

In [ ]:
from typing import Literal

Colour = Literal['W', 'B']
Move   = tuple[Colour, Square, Square]

The function `get_move` takes two states `before` and `after`, where `after` results from `before` by a single
jump, and returns this jump as a move.  `Old` and `New` are the squares of the knights of the colour that moves,
before and after the jump.  `source` is the only square in `Old` that is not in `New` and `target` is the only
square in `New` that is not in `Old`.

In [ ]:
def get_move(before: State, after: State) -> Move:
    (White1, Black1), (White2, Black2) = before, after
    colour: Colour
    if White1 != White2:
        colour, Old, New = 'W', White1, White2
    else:
        colour, Old, New = 'B', Black1, Black2
    source, = Old - New
    target, = New - Old
    return (colour, source, target)

In [ ]:
get_move(start, (frozenset({'b4', 'c2'}), frozenset({'a1', 'b3'})))

The function `describe_move` takes the same arguments as `get_move` and returns a short description of the jump,
e.g. `B c1-b3` if the black knight on `c1` jumps to `b3`.

In [ ]:
def describe_move(before: State, after: State) -> str:
    colour, source, target = get_move(before, after)
    return f'{colour} {source}-{target}'

The function `print_path` takes a solution `Path`, i.e. a list of states where every state results from its
predecessor by a single jump, and prints the boards of all states side by side, `per_row` boards in a row.
Every board is headed by the number and the description of the move leading to it.

In [ ]:
def print_path(Path: list[State], per_row: int = 6) -> None:
    Captions = ['start'] + [f'{i}. {describe_move(Path[i-1], Path[i])}' for i in range(1, len(Path))]
    Boards   = [board_lines(state) for state in Path]
    for k in range(0, len(Path), per_row):
        print('   '.join(caption.ljust(11) for caption in Captions[k:k+per_row]).rstrip())
        for line in range(len(Boards[0])):
            print('   '.join(board[line].ljust(11) for board in Boards[k:k+per_row]).rstrip())
        print()

In [ ]:
print_path(Path)

## Animating the Solution

Jupyter displays the output of a cell according to its MIME type.  Output of type `text/html` is rendered as a
part of the web page and the JavaScript code contained in it is executed, no matter which kernel created the
output.  In a notebook with a Python kernel, the class `HTML` from the module `IPython.display` creates such
output.  We use this to animate the solution in the browser: Python only computes the moves and the positions of the
squares in the drawing, while the animation itself is written in JavaScript.  The solution is animated twice:
on the board and on the graph of knight moves.

We need the module `json` to turn Python data into JavaScript literals, the module `uuid` to create unique
identifiers, and the class `HTML` from the module `IPython.display` to create HTML output.

In [ ]:
import json
import uuid
from IPython.display import HTML

An object of the class `Layout` describes how the squares are drawn:
- `kind` is `'board'` if the squares are drawn as the squares of the board and `'graph'` if they are drawn as
  the round nodes of the graph of knight moves,
- `width` and `height` are the width and the height of the drawing,
- `size` is the side length of a square or the diameter of a node; the knights are scaled accordingly,
- `centers` maps every square to the coordinates `(x, y)` of its centre in the drawing; as usual in computer
  graphics, the $y$-axis points downwards,
- `labels` is a list of triples `(text, x, y)`: every `text` is written centred at the point `(x, y)`, and
- `edges` is the list of the edges of the graph, every edge is a pair of squares; for the board, this list is
  empty.

The decorator `@dataclass` creates a constructor that takes the attributes in the order given here.  The
function `asdict` converts a `Layout` into a dictionary, which the module `json` can turn into a JavaScript
object.

In [ ]:
from dataclasses import dataclass, asdict

Point = tuple[float, float]

@dataclass
class Layout:
    kind:    Literal['board', 'graph']
    width:   float
    height:  float
    size:    float
    centers: dict[Square, Point]
    labels:  list[tuple[str, float, float]]
    edges:   list[tuple[Square, Square]]

The function `board_layout` returns the layout of the board.  Every square has the side length `size`.  To the
left of the board and below it, there is a strip of width `margin` for the numbers of the rows and the names of
the columns.  The local function `center` computes the centre of a `square` from the index `column` of its
column and the index `row` of its row, both counted from $0$.  As row $0$ is at the bottom, but the $y$-axis
points downwards, the $y$-coordinate of the centre is $(3.5 - \texttt{row}) \cdot \texttt{size}$.

In [ ]:
def board_layout(size: float = 60, margin: float = 24) -> Layout:
    def center(square: Square) -> Point:
        column = ord(square[0]) - ord('a')
        row    = int(square[1]) - 1
        return (margin + (column + 0.5) * size, (3.5 - row) * size)
    Centers = { s: center(s) for s in sorted(Squares) }
    Labels  = ([(c, margin + (i + 0.5) * size, 4 * size + margin / 2) for i, c in enumerate('abcd')] +
               [(r, margin / 2, (3.5 - i) * size)                    for i, r in enumerate('1234')])
    return Layout('board', margin + 4 * size, 4 * size + margin, size, Centers, Labels, [])

In the graph, the name of a square is written next to its node.  The function `label_position` takes a `square`,
the dictionary `Centers` of the centres of all nodes, and a `distance`.  It returns the point where the name of
`square` is written.  This point lies at the given `distance` from the centre of the node, preferably below it.
If an edge of the node leaves it in a direction that is close to this direction, the name is written above the
node instead, or to the left, or to the right of it.  These four directions are the unit vectors in the list
`Directions`, in the order of preference, where the $y$-axis points downwards.

The local function `is_free` takes a direction `d` and checks whether the angle between `d` and every edge of
the node is at least $30^\circ$, i.e. whether the cosine of this angle is at most $\cos(30^\circ)$.  The cosine
of the angle between two vectors is their dot product divided by the product of their lengths, and `d` has the
length $1$.  `label_position` takes the first direction in `Directions` that is free.  An edge can be close to
at most one of the four directions, and a node has at most three edges.  Hence, at least one direction is free.

In [ ]:
import math

Directions: list[Point] = [(0, 1), (0, -1), (-1, 0), (1, 0)]   # below, above, left, right

def label_position(square: Square, Centers: dict[Square, Point], distance: float) -> Point:
    x, y = Centers[square]
    def is_free(d: Point) -> bool:
        return all(((u - x) * d[0] + (v - y) * d[1]) / math.dist((x, y), (u, v)) <= math.cos(math.pi / 6)
                   for u, v in [Centers[t] for t in Jumps[square]])
    dx, dy = next(d for d in Directions if is_free(d))
    return (x + distance * dx, y + distance * dy)

The function `graph_layout` returns the layout of the graph of knight moves.  The nodes are placed where
`graphviz` places them in the drawing created by `board_graph`.  The method `pipe` with the format `'plain'`
returns a description of this drawing as text.  For every node, it contains a line like
```
node a1 2.5069 3.307 0.85775 0.5 "a1: B" filled ellipse black black
```
that starts with the word `node`, followed by the name of the node and the coordinates of its centre in inches,
where the $y$-axis points upwards.  `Inches` maps every square to these coordinates.  They are converted into the
coordinates of the animation: one inch becomes `scale` units, the $y$-axis is turned upside down, and around the
nodes there is a border of width `border`, which leaves room for the names of the squares.  The function
`label_position` places these names at a distance of $13$ units from the circles of the nodes.

The drawings of different states differ only in the labels of their nodes.  The positions of the nodes are the
same, except for a translation that depends on the widths of the labels.  As `xmin` and `ymax` are subtracted,
this translation does not matter and `board_graph` can be called with any state, e.g. with `start`.  The edges
are the pairs `(s, t)` of squares with `s < t` such that a knight can jump from `s` to `t`, as in `board_graph`.

In [ ]:
def graph_layout(size: float = 50, scale: float = 72) -> Layout:
    Lines   = board_graph(start).pipe(format='plain').decode().splitlines()
    Nodes   = [line.split() for line in Lines if line.startswith('node ')]
    Inches  = { node[1]: (float(node[2]), float(node[3])) for node in Nodes }
    xmin    = min(x for x, y in Inches.values())
    xmax    = max(x for x, y in Inches.values())
    ymin    = min(y for x, y in Inches.values())
    ymax    = max(y for x, y in Inches.values())
    border  = size / 2 + 24
    Centers = { s: (round(border + (x - xmin) * scale, 1), round(border + (ymax - y) * scale, 1))
                for s, (x, y) in Inches.items()
              }
    Labels  = [(s, *label_position(s, Centers, size / 2 + 13)) for s in sorted(Squares)]
    Edges   = [(s, t) for s in sorted(Squares) for t in sorted(Jumps[s]) if s < t]
    width   = round(2 * border + (xmax - xmin) * scale, 1)
    height  = round(2 * border + (ymax - ymin) * scale, 1)
    return Layout('graph', width, height, size, Centers, Labels, Edges)

The string `AnimationTemplate` contains the HTML and JavaScript code of the animation.  It draws the squares as
described by a layout, i.e. as the board or as the graph of knight moves, as an SVG graphic, and the knights as
the chess symbol ♞, filled white or black.  The buttons have the following effects:
- `▶ Play` plays the solution, one jump after the other.  While the animation plays, the button reads
  `⏸ Pause` and pauses the animation.
- `Step ▶` makes the next jump and `◀ Back` takes back the last jump.
- `⏮ Reset` returns to the start position.

The slider shows the position after any number of moves at once.  The source and the target square of the last
jump are highlighted and the line above the drawing describes this jump.  The code `&#xFE0E;` (`\uFE0E` in
JavaScript strings) after the symbols on the buttons asks the browser to draw them as text symbols, not as emoji.

In the graph, the squares are drawn as squares whose corners are rounded with a radius of half their side
length, i.e. as circles.  Therefore, the same code draws the squares of the board, the nodes of the graph, and
the highlighting of the last jump.  The edges of the graph are drawn first, so that the nodes cover their ends.
The nodes are coloured like the squares of the board.  So the graph shows that a knight always jumps from a
light square to a dark one or vice versa, and a jump along an edge in the graph is the same jump as on the board.

Whenever a knight lands on a square, the animation plays the sound of a chess piece that is put on the board.
The checkbox `Sound` switches the sound off and on.  The sound is generated by the function `clack` with the
*Web Audio API* of the browser and consists of two parts:
- a burst of noise that dies away within 25 milliseconds and is filtered by a band-pass filter: the click of
  the wood, and
- a low tone that dies away within 0.1 seconds: the resonance of the board.

Browsers play sounds only after the user has interacted with the page.  Therefore, the *audio context*, which
produces the sound, is created by the function `startAudio` when one of the buttons `▶ Play`, `Step ▶`, or
`◀ Back` is clicked for the first time.  The slider and the button `⏮ Reset` show a position at once, without
jumps, so they make no sound.

The template contains the following placeholders, which the function `animation` below replaces:
- `__ID__` is a unique identifier.  It is part of the `id` attributes of all HTML elements of the animation, so
  that the animations of different cells do not interfere.
- `__LAYOUT__` is the layout as a JavaScript object,
- `__VIEWBOX__` is the area of the drawing, i.e. `0 0 width height`,
- `__WHITE__` and `__BLACK__` are the lists of the squares of the white and the black knights at the start,
- `__MOVES__` is the list of moves, every move is a list `[colour, source, target]`,
- `__DURATION__` is the duration of a jump in milliseconds, and
- `__WIDTH__` is the width of the drawing on the page.

The JavaScript code looks up the elements of the animation only when they are part of the page.  Until then, the
function `init` calls itself again after 50 milliseconds.

In [ ]:
AnimationTemplate = r"""
<div id="knights-__ID__" style="font-family: sans-serif;">
  <style>
    #knights-__ID__ button { padding: 4px 10px; margin-right: 4px; cursor: pointer; font-size: 14px; }
    #knights-__ID__ input[type=range] { vertical-align: middle; width: 160px; }
    #knights-__ID__ label  { margin-left: 8px; font-size: 14px; }
  </style>
  <div>
    <button id="reset-__ID__">⏮&#xFE0E; Reset</button>
    <button id="back-__ID__">◀&#xFE0E; Back</button>
    <button id="play-__ID__">▶&#xFE0E; Play</button>
    <button id="step-__ID__">Step ▶&#xFE0E;</button>
    <input  id="slider-__ID__" type="range" min="0" value="0">
    <label><input id="sound-__ID__" type="checkbox" checked> Sound</label>
  </div>
  <div id="caption-__ID__" style="margin: 8px 0; font-size: 15px;"></div>
  <svg id="board-__ID__" width="__WIDTH__" viewBox="__VIEWBOX__"></svg>
</div>
<script>
(function () {
  const layout   = __LAYOUT__;    // the centres of the squares, the labels, and the edges
  const white    = __WHITE__;     // the squares of the white knights at the start
  const black    = __BLACK__;     // the squares of the black knights at the start
  const moves    = __MOVES__;     // the moves as triples [colour, source, target]
  const duration = __DURATION__;  // the duration of a jump in milliseconds
  const names    = { W: 'white', B: 'black' };
  const size     = layout.size;   // the side length of a square or the diameter of a node
  const radius   = layout.kind === 'graph' ? size / 2 : 0;   // the nodes of the graph are round
  const svgNS    = 'http://www.w3.org/2000/svg';

  function init() {
    const svg = document.getElementById('board-__ID__');
    if (svg === null) {           // the output is not yet part of the page
      setTimeout(init, 50);
      return;
    }
    const caption = document.getElementById('caption-__ID__');
    const slider  = document.getElementById('slider-__ID__');
    const play    = document.getElementById('play-__ID__');
    slider.max = moves.length;

    // returns the coordinates of the centre of a square
    function center(square) {
      return layout.centers[square];
    }
    // creates an SVG element with the given attributes and appends it to parent
    function create(tag, attributes, parent) {
      const element = document.createElementNS(svgNS, tag);
      for (const [key, value] of Object.entries(attributes)) {
        element.setAttribute(key, value);
      }
      parent.appendChild(element);
      return element;
    }
    // the edges of the graph, drawn first, so that the nodes cover their ends
    for (const [s, t] of layout.edges) {
      const [x1, y1] = center(s);
      const [x2, y2] = center(t);
      create('line', { x1: x1, y1: y1, x2: x2, y2: y2, stroke: '#7a5c3e', 'stroke-width': 2 }, svg);
    }
    // the squares or the nodes, coloured as in the picture of the puzzle: a1 is a light square
    for (const square of Object.keys(layout.centers)) {
      const [x, y] = center(square);
      const dark   = (square.charCodeAt(0) + Number(square[1])) % 2 === 1;
      create('rect', { x: x - size / 2, y: y - size / 2, width: size, height: size, rx: radius,
                       fill: dark ? '#b58863' : '#f0d9b5', stroke: '#7a5c3e',
                       'stroke-width': radius > 0 ? 1.5 : 0.5 }, svg);
    }
    // the labels: the names of the columns and the numbers of the rows, or the names of the nodes
    for (const [text, x, y] of layout.labels) {
      const label = create('text', { x: x, y: y, 'text-anchor': 'middle', 'dominant-baseline': 'central',
                                     'font-size': 14, fill: '#888' }, svg);
      label.textContent = text;
    }
    // the highlighting of the source and the target square of the last jump
    const marks = [0, 1].map(() => create('rect', { width: size, height: size, rx: radius,
                                                    fill: 'rgba(255, 215, 0, 0.5)', visibility: 'hidden' }, svg));
    // creates a knight: a group that contains the symbol ♞ centred at the origin
    function knight(colour) {
      const group = create('g', {}, svg);
      const glyph = create('text', { 'text-anchor': 'middle', 'dominant-baseline': 'central', 'font-size': 44 * size / 60,
                                     fill:   colour === 'W' ? '#ffffff' : '#000000',
                                     stroke: colour === 'W' ? '#000000' : '#ffffff',
                                     'stroke-width': 1.5, 'paint-order': 'stroke' }, group);
      glyph.textContent = '♞';
      return group;
    }
    const start = {};             // maps the squares of the start position to their knights
    for (const square of white) start[square] = knight('W');
    for (const square of black) start[square] = knight('B');

    let occupant   = {};          // maps every occupied square to its knight
    let current    = 0;           // the number of moves made so far
    let busy       = false;       // true while a knight jumps
    let playing    = false;       // true while the animation plays
    let timer      = null;        // the pending call of nextMove, if any
    let generation = 0;           // incremented in order to cancel a running jump

    const sound = document.getElementById('sound-__ID__');
    let audio   = null;           // the audio context, created at the first click on a button

    // creates the audio context, if necessary; must be called when the user clicks a button
    function startAudio() {
      const Context = window.AudioContext || window.webkitAudioContext;
      if (audio === null && Context) {
        audio = new Context();
      }
      if (audio !== null && audio.state === 'suspended') {
        audio.resume();
      }
    }
    // plays the sound of a chess piece that is put on the board
    function clack() {
      if (audio === null || !sound.checked) {
        return;
      }
      const now = audio.currentTime;
      // a burst of noise that dies away within 25 ms, filtered by a band pass: the click of the wood
      const length = Math.floor(0.025 * audio.sampleRate);
      const buffer = audio.createBuffer(1, length, audio.sampleRate);
      const data   = buffer.getChannelData(0);
      for (let i = 0; i < length; i++) {
        data[i] = (2 * Math.random() - 1) * Math.pow(1 - i / length, 3);
      }
      const noise  = audio.createBufferSource();
      noise.buffer = buffer;
      const filter = audio.createBiquadFilter();
      filter.type            = 'bandpass';
      filter.frequency.value = 2000;
      filter.Q.value         = 1.2;
      const noiseGain = audio.createGain();
      noiseGain.gain.value = 0.7;
      noise.connect(filter);
      filter.connect(noiseGain);
      noiseGain.connect(audio.destination);
      // a low tone that dies away within 0.1 s: the resonance of the board
      const tone = audio.createOscillator();
      tone.frequency.setValueAtTime(300, now);
      tone.frequency.exponentialRampToValueAtTime(150, now + 0.1);
      const toneGain = audio.createGain();
      toneGain.gain.setValueAtTime(0.4, now);
      toneGain.gain.exponentialRampToValueAtTime(0.001, now + 0.1);
      tone.connect(toneGain);
      toneGain.connect(audio.destination);
      noise.start(now);
      tone.start(now);
      tone.stop(now + 0.1);
    }

    // draws a knight centred at (x, y), enlarged by the factor scale
    function place(group, x, y, scale) {
      group.setAttribute('transform', `translate(${x}, ${y}) scale(${scale})`);
    }
    // updates the slider, the caption, and the highlighted squares
    function update() {
      slider.value = current;
      if (current === 0) {
        caption.textContent = 'Start position';
        marks.forEach(mark => mark.setAttribute('visibility', 'hidden'));
        return;
      }
      const [colour, source, target] = moves[current - 1];
      caption.textContent = `Move ${current} of ${moves.length}: ${names[colour]} knight ${source} → ${target}`;
      if (current === moves.length) {
        caption.textContent += ' (solved)';
      }
      [source, target].forEach((square, i) => {
        const [x, y] = center(square);
        marks[i].setAttribute('x', x - size / 2);
        marks[i].setAttribute('y', y - size / 2);
        marks[i].setAttribute('visibility', 'visible');
      });
    }
    // shows the position after k moves without animation
    function setPosition(k) {
      generation++;
      busy     = false;
      occupant = Object.assign({}, start);
      for (const [colour, source, target] of moves.slice(0, k)) {
        occupant[target] = occupant[source];
        delete occupant[source];
      }
      for (const [square, group] of Object.entries(occupant)) {
        const [x, y] = center(square);
        place(group, x, y, 1);
      }
      current = k;
      update();
    }
    // lets the knight on source jump to target and adds delta to current afterwards;
    // the knight grows during the first half of the jump and shrinks during the second half
    function jump(source, target, delta) {
      const group = occupant[source];
      delete occupant[source];
      occupant[target] = group;
      svg.appendChild(group);       // the jumping knight is drawn on top
      const [x0, y0] = center(source);
      const [x1, y1] = center(target);
      const g = generation;
      let t0  = null;
      busy    = true;
      function frame(time) {
        if (g !== generation) {     // the jump has been cancelled
          return;
        }
        if (t0 === null) {
          t0 = time;
        }
        const t = Math.min((time - t0) / duration, 1);
        const s = t * t * (3 - 2 * t);   // starts and stops smoothly
        place(group, x0 + (x1 - x0) * s, y0 + (y1 - y0) * s, 1 + 0.4 * Math.sin(Math.PI * t));
        if (t < 1) {
          requestAnimationFrame(frame);
        } else {
          busy     = false;
          current += delta;
          clack();
          update();
          if (playing) {
            timer = setTimeout(nextMove, duration / 2);
          }
        }
      }
      requestAnimationFrame(frame);
    }
    function stop() {
      playing = false;
      clearTimeout(timer);
      play.textContent = '▶\uFE0E Play';
    }
    // makes the next move while the animation plays
    function nextMove() {
      timer = null;
      if (!playing) {
        return;
      }
      if (current === moves.length) {
        stop();
        return;
      }
      const [colour, source, target] = moves[current];
      jump(source, target, 1);
    }
    play.onclick = () => {
      startAudio();
      if (playing) {
        stop();
        return;
      }
      if (current === moves.length) {
        setPosition(0);
      }
      playing = true;
      play.textContent = '⏸\uFE0E Pause';
      if (!busy) {
        nextMove();
      }
    };
    document.getElementById('step-__ID__').onclick = () => {
      startAudio();
      stop();
      if (!busy && current < moves.length) {
        const [colour, source, target] = moves[current];
        jump(source, target, 1);
      }
    };
    document.getElementById('back-__ID__').onclick = () => {
      startAudio();
      stop();
      if (!busy && current > 0) {
        const [colour, source, target] = moves[current - 1];
        jump(target, source, -1);
      }
    };
    document.getElementById('reset-__ID__').onclick = () => {
      stop();
      setPosition(0);
    };
    slider.oninput = () => {
      stop();
      setPosition(Number(slider.value));
    };
    setPosition(0);
  }
  init();
})();
</script>
"""

The function `animation` takes a solution `Path`, a `layout`, the `duration` of a jump in milliseconds, and the
`width` of the drawing on the page.  It computes the moves of the solution, replaces the placeholders of `AnimationTemplate`,
and returns the resulting HTML object.  When this object is the value of a cell, Jupyter displays the animation.
The module `json` can't convert frozen sets, so the squares of the knights at the start are converted into
sorted lists first.

In [ ]:
def animation(Path: list[State], layout: Layout, duration: int = 500, width: str = '320px') -> HTML:
    White, Black = Path[0]
    Moves  = [get_move(Path[i-1], Path[i]) for i in range(1, len(Path))]
    Values = { '__ID__':       uuid.uuid4().hex,
               '__LAYOUT__':   json.dumps(asdict(layout)),
               '__VIEWBOX__':  f'0 0 {layout.width} {layout.height}',
               '__WHITE__':    json.dumps(sorted(White)),
               '__BLACK__':    json.dumps(sorted(Black)),
               '__MOVES__':    json.dumps(Moves),
               '__DURATION__': str(duration),
               '__WIDTH__':    width
             }
    html = AnimationTemplate
    for placeholder, value in Values.items():
        html = html.replace(placeholder, value)
    return HTML(html)

JupyterLab executes the JavaScript code of an output only if the output has been created in the current session
or if the notebook is trusted.  If the buttons do not work after the notebook has been reopened, the cells that
display the animations have to be executed again.

In [ ]:
animation(Path, board_layout())

The second animation shows the same solution on the graph of knight moves, where every jump of a knight follows
an edge.  It shows that the knights never pass each other on the path from `b4` to `b2`.  They change their order
only by stepping aside into the dead end `c1`.

In [ ]:
animation(Path, graph_layout(), width='640px')